In [ ]:
# @title Set ONLINE=False if running locally
ONLINE = True

In [ ]:
# @title Imports
import pandas as pd
import numpy as np
import math
import gdown
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm

In [ ]:
# Download training and submission data
if ONLINE:
    url = 'https://drive.google.com/file/d/1lDEtOq644qlaw_vh0i5c3ChVfrHJiuMc/view?usp=sharing'
    output = 'data_train.csv'
    gdown.download(url, output, fuzzy=True)

In [ ]:
# load training data
number_of_users, number_of_movies = (10000, 1000)
data_pd = pd.read_csv('data_train.csv')

# Split the dataset into train and test
train_size = 0.9
train_pd, test_pd = train_test_split(data_pd, train_size=train_size, random_state=42)

In [ ]:
def extract_users_items_predictions(data_pd):
    users, movies = \
        [np.squeeze(arr) for arr in np.split(data_pd.Id.str.extract('r(\d+)_c(\d+)').values.astype(int) - 1, 2, axis=-1)]
    predictions = data_pd.Prediction.values
    return users, movies, predictions

In [ ]:
# extract matrix data
train_users, train_movies, train_predictions = extract_users_items_predictions(train_pd)
test_users, test_movies, test_predictions = extract_users_items_predictions(test_pd)

In [ ]:
# create full matrix of observed values
data = np.full((number_of_users, number_of_movies), 0)
mask = np.full((number_of_users, number_of_movies), 0) # 0 -> unobserved value, 1->observed value

for user, movie, pred in zip(train_users, train_movies, train_predictions):
    data[user][movie] = pred
    mask[user][movie] = 1

In [ ]:
# define RMSE metric
rmse = lambda x, y: math.sqrt(mean_squared_error(x, y))

def get_score(predictions, target_values=test_predictions):
    return rmse(predictions, target_values)

In [ ]:
# @title Use GPU in colab: Runtime->Change Runtime type

#  use gpu if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('Using device:', device)

In [ ]:
# @title Model definition
class Encoder(nn.Module):
    def __init__(self, input_dimension, hidden_dimension=128, encoded_dimension=64, drop_p=0.1, hidden_layers=0, activation=nn.SELU()):
        super().__init__()

        modules = []

        # input layer
        modules.append(nn.Linear(in_features=input_dimension, out_features=hidden_dimension))
        modules.append(activation)

        # hidden layers
        c = hidden_dimension - encoded_dimension
        for i in range(hidden_layers):
            modules.append(nn.Linear(in_features=encoded_dimension + c//(2**i), out_features=encoded_dimension + c//(2**(i+1))))
            modules.append(activation)

        # output layer
        modules.append(nn.Linear(in_features=encoded_dimension + c//(2**hidden_layers), out_features=encoded_dimension))
        modules.append(activation)

        # dropout layer
        modules.append(nn.Dropout(drop_p))

        self.model = nn.Sequential(*modules)

    def forward(self, data):
        return self.model(data)

class Decoder(nn.Module):
    def __init__(self, output_dimension, hidden_dimension=128, encoded_dimension=16, drop_p=0.1, hidden_layers=1, activation=nn.SELU()):
        super().__init__()

        modules = []

        c = hidden_dimension - encoded_dimension
        # input layer
        modules.append(nn.Linear(in_features=encoded_dimension, out_features=encoded_dimension + c//(2**hidden_layers)))
        modules.append(activation)

        # hidden layers
        for i in range(hidden_layers, 0, -1):
            modules.append(nn.Linear(in_features=encoded_dimension + c//(2**i), out_features=encoded_dimension + c//(2**(i-1))))
            modules.append(activation)

        # output layer
        modules.append(nn.Linear(in_features=hidden_dimension, out_features=output_dimension))
        modules.append(activation)

        self.model = nn.Sequential(*modules)

    def forward(self, encodings):
        return self.model(encodings)

class AutoEncoder(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, data, encode):
        if encode:
            return self.encoder(data)
        else:
            return self.decoder(data)

In [ ]:
class MLP(nn.Module):

    def __init__(self, input_dimension, hidden_dimension, output_dimension, activation=nn.SELU()):
        super().__init__()

        self.model = nn.Sequential (
            nn.Linear(in_features=input_dimension, out_features=hidden_dimension),
            activation,
            nn.Linear(in_features=hidden_dimension, out_features=output_dimension),
            activation,
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
class CFDA(nn.Module):
    def __init__(self, autoencoder_users, autoencoder_movies, predictor):
        super().__init__()
        self.autoencoder_users = autoencoder_users
        self.autoencoder_movies = autoencoder_movies
        self.predictor=predictor

    def forward(self, user, movie):
        movie_embedding = self.autoencoder_movies(movie, True)
        user_embedding = self.autoencoder_users(user, True)
        full_embedding = torch.cat((user_embedding, movie_embedding), dim=1)
        movie_reconstructed = self.autoencoder_movies(movie_embedding, False)
        user_reconstructed = self.autoencoder_users(user_embedding, False)
        return self.predictor(full_embedding), user_reconstructed, movie_reconstructed


In [ ]:
# @title Hyperparameters
encoded_dimension = 32
hidden_dimension = 64
drop_p = 0.1
activation = nn.SELU()
hidden_layers_encoder = 1
hidden_layers_decoder = 1

batch_size = 4096
num_epochs = 100
show_validation_score_every_epochs = 1
learning_rate = 0.001
weight_decay = 0
patience=3

In [ ]:
autoencoder_users = AutoEncoder(
    encoder=Encoder(
        input_dimension=number_of_movies,
        hidden_dimension=hidden_dimension,
        encoded_dimension=encoded_dimension,
        drop_p=drop_p,
        hidden_layers=hidden_layers_encoder,
        activation=activation,
    ),
    decoder=Decoder(
        output_dimension=number_of_movies,
        hidden_dimension=hidden_dimension,
        encoded_dimension=encoded_dimension,
        drop_p=drop_p,
        hidden_layers=hidden_layers_decoder,
        activation=activation,
    ),
).to(device)

autoencoder_movies = AutoEncoder(
    encoder=Encoder(
        input_dimension=number_of_users,
        hidden_dimension=hidden_dimension,
        encoded_dimension=encoded_dimension,
        drop_p=drop_p,
        hidden_layers=hidden_layers_encoder,
        activation=activation,
    ),
    decoder=Decoder(
        output_dimension=number_of_users,
        hidden_dimension=hidden_dimension,
        encoded_dimension=encoded_dimension,
        drop_p=drop_p,
        hidden_layers=hidden_layers_decoder,
        activation=activation,
    ),
).to(device)

predictor = MLP(
    input_dimension=encoded_dimension*2,
    hidden_dimension=encoded_dimension//2,
    output_dimension=1,
    activation=nn.SELU()
).to(device)

model = CFDA(
    autoencoder_users=autoencoder_users,
    autoencoder_movies=autoencoder_movies,
    predictor=predictor
).to(device)

In [ ]:
# @title optimizer, scheduler, loss functions
optimizer = optim.Adam(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    'min',
    patience=patience,
    factor=0.5,
)

l = nn.MSELoss().to(device)
def loss_function_mlp(original, reconstructed):
    return l(original, reconstructed)

def loss_function_autoencoder(original, reconstructed, mask):
    return torch.mean(mask * (original - reconstructed) ** 2)

In [ ]:
# @title Dataloaders
data_torch = torch.tensor(data, device=device).float()
data_torchT = data_torch.T
mask_torch = torch.tensor(mask, device=device)
mask_torchT = mask_torch.T
train_users_torch = torch.tensor(train_users, device=device)
train_movies_torch = torch.tensor(train_movies, device=device)
train_predictions_torch = torch.tensor(
    train_predictions, device=device, dtype=torch.float32)

dataloader = DataLoader(
    TensorDataset(train_users_torch, train_movies_torch, train_predictions_torch),
    batch_size=batch_size,
    shuffle=True
)

test_users_torch = torch.tensor(test_users, device=device)
test_movies_torch = torch.tensor(test_movies, device=device)
dataloader_test = DataLoader(
    TensorDataset(test_users_torch, test_movies_torch),
    batch_size=4096,
)

In [ ]:
# @title Training
from copy import deepcopy

best_error = math.inf
best_model_state = None

step = 0
with tqdm(total=len(dataloader) * num_epochs) as pbar:
    for epoch in range(num_epochs):
        for user_id_batch, movie_id_batch, predictions_batch in dataloader:
            optimizer.zero_grad()

            movies = data_torchT[movie_id_batch]
            users = data_torch[user_id_batch]
            reconstructed_batch, reconstructed_users_batch, reconstructed_movies_batch = model(users, movies)

            loss = loss_function_mlp(predictions_batch.unsqueeze(1), reconstructed_batch)
            loss += loss_function_autoencoder(user, reconstructed_users_batch, mask_torch[user_id_batch])
            loss += loss_function_autoencoder(movie, reconstructed_movies_batch, mask_torchT[movie_id_batch])

            loss.backward()

            optimizer.step()

            pbar.update(1)
            step += 1

        if epoch % show_validation_score_every_epochs == 0:
            model.eval()
            predictions = []
            for user_id_batch, movies_id_batch in dataloader_test:
                p, _, _ = model(data_torch[user_id_batch], data_torchT[movies_id_batch])
                predictions.append(p.detach().cpu().numpy().clip(1, 5))
            predictions = np.vstack(predictions)
            reconstruction_rmse = get_score(predictions)
            if reconstruction_rmse < best_error:
                best_error = reconstruction_rmse
                if best_error < 0.9830:
                    best_model_state = deepcopy(model.state_dict())
            pbar.set_description('At epoch {:3d} loss is {:.4f}, best loss {:.4f}'.format(epoch + 1, reconstruction_rmse, best_error))
            model.train()
        scheduler.step(reconstruction_rmse)